In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import matplotlib.pyplot as plt

# Hi-COLA vs COLA_NM4096

Matter power spectrum and halo mass function of the Hi-COLA runs matched to Bart's COLA_NM4096
simulations (10 boxes of 1024 Mpc/h, Euclid cosmology), for ΛCDM and nDGP.
The reference is the mean over its boxes, with the box-to-box standard deviation as 1σ band.

Everything below follows the parameters in the next cell. Hi-COLA measurements are read from
the run folder on scratch, or from the kept copy in `hicola_data/hicola/` once scratch has been
cleaned up (see `PIPELINE.md`).

In [ ]:
target = 'COLA_NM4096'
redshift = 1.65              # 1.65, 1.4, 1.2 or 1.0
models = {                   # model -> panel title; LCDM first (it is the denominator of nDGP/LCDM)
    'LCDM': r'$\Lambda\mathrm{CDM}$',
    'nDGP_N1.2': r'$\mathrm{nDGP}$',
}
runs = {                     # Hi-COLA runs to show: label -> realizations averaged together
    'COLA3072 (this work)': ['s1234'],
    # 'COLA3072 (paired-fixed)': list(an.PAIRED_FIXED),
}
reference_name = 'COLA4096'
reference_boxes = list(range(1, 11))  # Bart's boxes averaged for the reference mean and 1σ
M_min = 1e12                 # lowest halo mass shown
pk_ratio_ylim = None         # y range of the nDGP/LCDM P(k) figure, e.g. (0.98, 1.1)

In [ ]:
# Figure styles: SMALL for multi-panel figures, BIG for one-column figures (shrunk more in a document)
SMALL = dict(figsize=(6.6, 4.35), label=10.5, tick=8.8, legend=8.3, lw=1.45)
BIG = dict(figsize=(6.6, 4.6), label=14.5, tick=12.5, legend=11.5, lw=1.9)
BAND_ALPHA = 0.18
REFERENCE_COLOR = '#1f4e79'
RATIO_REF_COLOR = '0.35'
RUN_COLORS = ['#c47f2b', '#4c8c4a', '#8e5ea2', '#b5446e']
RUN_STYLES = ['--', '-.', ':', (0, (5, 1))]
K_LABEL = r'$k\,[h\,\mathrm{Mpc}^{-1}]$'
M_LABEL = r'$M_{200c}\,[h^{-1}M_\odot]$'


def run_style(i):
    return dict(color=RUN_COLORS[i % len(RUN_COLORS)], ls=RUN_STYLES[i % len(RUN_STYLES)])


def finish(axes, s):
    for ax in np.atleast_1d(axes).ravel():
        if not ax.get_visible():
            continue
        ax.minorticks_on()
        ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
        scale = s['tick'] / SMALL['tick']
        ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=s['tick'], length=3.4 * scale, width=0.7 * scale)
        ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=s['tick'] - 0.8, length=2.0 * scale, width=0.55 * scale)
        for spine in ax.spines.values():
            spine.set_linewidth(0.7 * scale)


def legend(ax, s, **kwargs):
    ax.legend(fontsize=s['legend'], frameon=False, handlelength=1.6, borderaxespad=0.3, labelspacing=0.3, **kwargs)

## Load

In [ ]:
lcdm, ndgp = models

# Reference: (kind, model) -> (x, mean, std over boxes); (kind, 'ratio') -> nDGP/LCDM per box, mean and std
ref = {}
for kind in ('PS', 'HMF'):
    for model in models:
        ref[kind, model] = an.load_reference(target, model, redshift, kind, boxes=reference_boxes)
    ref[kind, 'ratio'] = an.reference_ratio(target, ndgp, lcdm, redshift, kind, boxes=reference_boxes)

# Hi-COLA measurements: (kind, model, run label) -> (x, y). Runs that do not exist (yet) are skipped.
hic = {}
for kind in ('PS', 'HMF'):
    for label, realizations in runs.items():
        for model in models:
            try:
                hic[kind, model, label] = an.load_hicola(target, model, realizations, redshift, kind)
            except FileNotFoundError as error:
                print(f'Skipping {label} / {model} / {kind}: {error}')

## Matter power spectrum

In [ ]:
# k^{3/2} P(k) against the reference (left) and the ratio to the reference mean (right), one row per model
s = SMALL
fig, axes = plt.subplots(2, 2, figsize=s['figsize'], dpi=300, constrained_layout=True)
for (ax_p, ax_r), (model, title) in zip(axes, models.items()):
    k_ref, P_ref, P_std = ref['PS', model]
    P_low, P_high = np.maximum(P_ref - P_std, 1e-30), P_ref + P_std
    ax_p.semilogx(k_ref, k_ref**1.5 * P_ref, lw=s['lw'], color=REFERENCE_COLOR, label=f'{reference_name} (mean)')
    ax_p.fill_between(k_ref, k_ref**1.5 * P_low, k_ref**1.5 * P_high, color=REFERENCE_COLOR, alpha=BAND_ALPHA,
                      label=rf'{reference_name} (1$\sigma$)')
    for i, label in enumerate(runs):
        if ('PS', model, label) not in hic:
            continue
        k, P = hic['PS', model, label]
        keep = an.within(k, k_ref.min(), k_ref.max())
        ax_p.semilogx(k[keep], k[keep]**1.5 * P[keep], lw=s['lw'], label=label, **run_style(i))
        ax_r.semilogx(k_ref, an.ratio(k, P, k_ref, P_ref), lw=s['lw'], label=label, **run_style(i))
        ax_r.fill_between(k_ref, an.ratio(k, P, k_ref, P_high), an.ratio(k, P, k_ref, P_low),
                          color=run_style(i)['color'], alpha=BAND_ALPHA)
    ax_r.axhline(1.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    ax_p.set_ylabel(r'$k^{3/2} P(k)$', fontsize=s['label'])
    ax_r.set_ylabel(rf'$P / P_\mathrm{{{reference_name}}}$', fontsize=s['label'])
    for ax in (ax_p, ax_r):
        ax.set_xlabel(K_LABEL, fontsize=s['label'])
        ax.set_title(title, fontsize=s['label'])
    legend(ax_p, s)
finish(axes, s)
plt.show()

In [ ]:
# nDGP / LCDM, one-column figure
s = BIG
k_ref, ratio_mean, ratio_std = ref['PS', 'ratio']
fig, ax = plt.subplots(figsize=s['figsize'], dpi=300, constrained_layout=True)
ax.semilogx(k_ref, ratio_mean, lw=s['lw'], color=REFERENCE_COLOR, label=f'{reference_name} (mean)')
ax.fill_between(k_ref, ratio_mean - ratio_std, ratio_mean + ratio_std, color=REFERENCE_COLOR, alpha=BAND_ALPHA,
                label=rf'{reference_name} (1$\sigma$)')
for i, label in enumerate(runs):
    if ('PS', lcdm, label) not in hic or ('PS', ndgp, label) not in hic:
        continue
    k_l, P_l = hic['PS', lcdm, label]
    k_n, P_n = hic['PS', ndgp, label]
    keep = an.within(k_l, k_ref.min(), k_ref.max())
    ax.semilogx(k_l[keep], an.ratio(k_n, P_n, k_l, P_l)[keep], lw=s['lw'], label=label, **run_style(i))
ax.axhline(1.0, color='0.4', lw=1.1, ls='--')
ax.set_xlabel(K_LABEL, fontsize=s['label'])
ax.set_ylabel(r'$P_{\mathrm{nDGP}} / P_{\mathrm{LCDM}}$', fontsize=s['label'])
if pk_ratio_ylim:
    ax.set_ylim(*pk_ratio_ylim)
legend(ax, s, loc='upper left')
finish(ax, s)
plt.show()

In [ ]:
# Distance to the reference in units of its box-to-box scatter, (P - mean) / sigma
s = SMALL
fig, axes = plt.subplots(1, 2, figsize=(s['figsize'][0], 2.6), dpi=300, constrained_layout=True, sharey=True)
for ax, (model, title) in zip(axes, models.items()):
    k_ref, P_ref, P_std = ref['PS', model]
    for i, label in enumerate(runs):
        if ('PS', model, label) not in hic:
            continue
        k, P = hic['PS', model, label]
        ax.semilogx(k_ref, (an.interp_to(k_ref, k, P) - P_ref) / P_std, lw=s['lw'], label=label, **run_style(i))
    for level in (-1, 1):
        ax.axhline(level, color=RATIO_REF_COLOR, lw=0.7, ls=':')
    ax.axhline(0, color=RATIO_REF_COLOR, lw=0.9)
    ax.set_xlabel(K_LABEL, fontsize=s['label'])
    ax.set_title(title, fontsize=s['label'])
axes[0].set_ylabel(r'$(P - P_\mathrm{mean}) / \sigma$', fontsize=s['label'])
legend(axes[0], s)
finish(axes, s)
plt.show()

## Halo mass function

In [ ]:
# Cumulative HMF against the reference (left) and the ratio to the reference mean (right).
# No band on the ratio: in the high-mass tail std > mean in some bins, so mean - std hits the floor.
s = SMALL
fig, axes = plt.subplots(2, 2, figsize=s['figsize'], dpi=300, constrained_layout=True)
for (ax_n, ax_r), (model, title) in zip(axes, models.items()):
    M_ref, n_ref, n_std = ref['HMF', model]
    keep_ref = (n_ref > 0) & (M_ref >= M_min)
    ax_n.loglog(M_ref[keep_ref], n_ref[keep_ref], lw=s['lw'], color=REFERENCE_COLOR, label=f'{reference_name} (mean)')
    ax_n.fill_between(M_ref[keep_ref], np.maximum(n_ref - n_std, 1e-30)[keep_ref], (n_ref + n_std)[keep_ref],
                      color=REFERENCE_COLOR, alpha=BAND_ALPHA, label=rf'{reference_name} (1$\sigma$)')
    for i, label in enumerate(runs):
        if ('HMF', model, label) not in hic:
            continue
        M, n = hic['HMF', model, label]
        keep = (n > 0) & (M >= M_min)
        ax_n.loglog(M[keep], n[keep], lw=s['lw'], label=label, **run_style(i))
        r = an.ratio(M, n, M_ref, n_ref)
        keep_r = np.isfinite(r) & (M_ref >= M_min)
        ax_r.semilogx(M_ref[keep_r], r[keep_r], lw=s['lw'], label=label, **run_style(i))
    ax_r.axhline(1.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    ax_n.set_ylim(1e-10, 1e-2)
    ax_n.set_ylabel(r'$n(>M)\,[h^3\,\mathrm{Mpc}^{-3}]$', fontsize=s['label'])
    ax_r.set_ylabel(rf'$n / n_\mathrm{{{reference_name}}}$', fontsize=s['label'])
    for ax in (ax_n, ax_r):
        ax.set_xlabel(M_LABEL, fontsize=s['label'])
        ax.set_title(title, fontsize=s['label'])
    legend(ax_n, s)
finish(axes, s)
plt.show()

In [ ]:
# nDGP / LCDM HMF, one-column figure
s = BIG
M_ref, ratio_mean, ratio_std = ref['HMF', 'ratio']
keep_ref = np.isfinite(ratio_mean) & (M_ref >= M_min)
fig, ax = plt.subplots(figsize=s['figsize'], dpi=300, constrained_layout=True)
ax.semilogx(M_ref[keep_ref], ratio_mean[keep_ref], lw=s['lw'], color=REFERENCE_COLOR, label=f'{reference_name} (mean)')
ax.fill_between(M_ref[keep_ref], (ratio_mean - ratio_std)[keep_ref], (ratio_mean + ratio_std)[keep_ref],
                color=REFERENCE_COLOR, alpha=BAND_ALPHA, label=rf'{reference_name} (1$\sigma$)')
for i, label in enumerate(runs):
    if ('HMF', lcdm, label) not in hic or ('HMF', ndgp, label) not in hic:
        continue
    M_l, n_l = hic['HMF', lcdm, label]
    M_n, n_n = hic['HMF', ndgp, label]
    r = an.ratio(M_n, n_n, M_l, n_l)
    keep = np.isfinite(r) & (M_l >= M_min)
    ax.semilogx(M_l[keep], r[keep], lw=s['lw'], label=label, **run_style(i))
ax.axhline(1.0, color='0.4', lw=1.1, ls='--')
ax.set_xlabel(M_LABEL, fontsize=s['label'])
ax.set_ylabel(r'$n_{\mathrm{nDGP}}(>M) / n_{\mathrm{LCDM}}(>M)$', fontsize=s['label'])
legend(ax, s)
finish(ax, s)
plt.show()